In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import paths as P

# 0. Draft Filtering

In [ ]:
import pandas as pd

def clean_data(df):
    df = df.drop(columns=[
        ('Assay ID', 'IEDB IRI'), 
        ('Reference', 'IEDB IRI'), 
        ('Reference', 'Type'), 
        ('Reference', 'PMID'), 
        ('Reference', 'Submission ID'), 
        ('Reference', 'Authors'), 
        ('Reference', 'Journal'),
        ('Reference', 'Title'), 
        ('Epitope', 'Epitope IRI'), 
        ('Epitope', 'Starting Position'), 
        ('Epitope', 'Ending Position'), 
        ('Epitope', 'IRI'), 
        ('Epitope', 'Synonyms'), 
        ('Epitope', 'Source Molecule IRI'), 
        ('Epitope', 'Molecule Parent IRI'), 
        ('Epitope', 'Source Organism IRI'), 
        ('Epitope', 'Species IRI'), 
        ('Epitope', 'Comments'), 
        ('Related Object', 'Epitope Relation'), 
        ('Related Object', 'Object Type'), 
        ('Related Object', 'Name'), 
        ('Related Object', 'Starting Position'), 
        ('Related Object', 'Ending Position'), 
        ('Related Object', 'IRI'), 
        ('Related Object', 'Synonyms'), 
        ('Related Object', 'Source Molecule'), 
        ('Related Object', 'Source Molecule IRI'), 
        ('Related Object', 'Molecule Parent'), 
        ('Related Object', 'Molecule Parent IRI'), 
        ('Related Object', 'Source Organism'), 
        ('Related Object', 'Source Organism IRI'), 
        ('Related Object', 'Species'), 
        ('Related Object', 'Species IRI'), 
        ('Host', 'Geolocation'), 
        ('Host', 'Geolocation IRI'), 
        ('Host', 'Sex'), 
        ('Host', 'Age'), 
        ('in vivo Antigen', 'Epitope Relation'), 
        ('in vivo Antigen', 'Object Type'), 
        ('in vivo Antigen', 'Name'), 
        ('in vivo Antigen', 'Reference Name'), 
        ('in vivo Antigen', 'Starting Position'), 
        ('in vivo Antigen', 'Ending Position'), 
        ('in vivo Antigen', 'IRI'), 
        ('in vivo Antigen', 'Source Molecule'), 
        ('in vivo Antigen', 'Source Molecule IRI'), 
        ('in vivo Antigen', 'Molecule Parent'), 
        ('in vivo Antigen', 'Molecule Parent IRI'), 
        ('in vivo Antigen', 'Source Organism'), 
        ('in vivo Antigen', 'Source Organism IRI'), 
        ('in vivo Antigen', 'Species'), 
        ('in vivo Antigen', 'Species IRI'), 
        ('in vivo Antigen', 'Adjuvants'), 
        ('in vivo Antigen', 'Route'), 
        ('in vivo Antigen', 'Dose Schedule'), 
        ('In vitro Process', 'Process Type'), 
        ('In vitro Process', 'Epitope Relation'), 
        ('In vitro Process', 'Object Type'), 
        ('In vitro Process', 'Name'), 
        ('In vitro Process', 'Reference Name'), 
        ('In vitro Process', 'Starting Position'), 
        ('In vitro Process', 'Ending Position'), 
        ('In vitro Process', 'IRI'), 
        ('In vitro Process', 'Source Molecule'), 
        ('In vitro Process', 'Source Molecule IRI'), 
        ('In vitro Process', 'Molecule Parent'), 
        ('In vitro Process', 'Molecule Parent IRI'), 
        ('In vitro Process', 'Source Organism'), 
        ('In vitro Process', 'Source Organism IRI'), 
        ('In vitro Process', 'Species'), 
        ('In vitro Process', 'Species IRI'), 
        ('MHC Restriction', 'IRI'), 
        ('MHC Restriction', 'Evidence IRI'), 
        ('Host', 'IRI'), 
        ('in vivo Process', 'Disease IRI'), 
        ('Assay', 'IRI'), 
        ('Antigen Presenting Cell', 'Source Tissue IRI'), 
        ('Antigen Presenting Cell', 'IRI'),
        ####
        ('Antigen Presenting Cell', 'Culture Condition'),
        ('MHC Restriction', 'Evidence Code'),
        ('Assay', 'Comments'),
        ('Assay', 'Location of Assay Data in Reference'),
        ('Assay', 'PDB ID'),
        ('Antigen Processing', 'Comments'),
        ('Epitope', 'Reference Name'),
        ])

    df = df[(~df[('MHC Restriction', 'Name')].str.contains("mutant", regex=False, na=False, case=False))]
    df = df[~df[('Epitope', 'Name')].str.contains("+", regex=False, na=False, case=False)]
    df = df[df[('Epitope', 'Object Type')] == "Linear peptide"]
    df = df.drop(columns=[
        ('Epitope', 'Object Type'), 
        ('Epitope', 'Modified residues'), 
        ('Epitope', 'Modifications')])

    df = df[df[('MHC Restriction', 'Class')] == "II"]
    df = df.drop(columns=[('MHC Restriction', 'Class')])
    return df

# df = pd.read_csv(P.IEDB_EXPORT, header=[0 , 1], low_memory=False)
# df_draft = clean_data(df.copy())
# df_draft.to_csv(r'draft.csv', index=False)
df_draft = pd.read_csv(P.DRAFT, header=[0 , 1], low_memory=False)

In [ ]:
def clean_data_human(df):
    df = df[(df[('MHC Restriction', 'Name')].str.contains("*", regex=False, na=False, case=False)) & (df[('MHC Restriction', 'Name')].str.startswith("HLA", na=False))]
    df = df[df[('MHC Restriction', 'Name')].str.startswith("HLA")]
    return df
def clean_data_nonhuman(df):
    df = df[~df[('MHC Restriction', 'Name')].str.startswith("HLA")]
    df = df[((~df[('MHC Restriction', 'Name')].str.contains("class", regex=False, na=False, case=False)))]
    df = df[((~df[('MHC Restriction', 'Name')].str.contains("Patr", regex=False, na=False, case=False)))]
    df = df[((df[('MHC Restriction', 'Name')].str.contains("H2", regex=False, na=False, case=False)))]
    df = df[((~df[('MHC Restriction', 'Name')].str.contains("H2-IAp", regex=False, na=False, case=False)) & ~(df[('MHC Restriction', 'Name')].str.contains("H2-IAr", regex=False, na=False, case=False))& ~(df[('MHC Restriction', 'Name')].str.contains("H2-IEb", regex=False, na=False, case=False))& ~(df[('MHC Restriction', 'Name')].str.contains("H2-IEg7", regex=False, na=False, case=False))& ~(df[('MHC Restriction', 'Name')].str.contains("H2-IEp", regex=False, na=False, case=False))& ~(df[('MHC Restriction', 'Name')].str.contains("H2-IEr", regex=False, na=False, case=False)))]
    return df
df_clean_human = clean_data_human(df_draft.copy())
df_clean_nonhuman = clean_data_nonhuman(df_draft.copy())
df_clean = pd.concat([df_clean_human, df_clean_nonhuman], ignore_index=True)
df_clean

In [ ]:
df = df_clean.copy()
df['name_len'] = df[('Epitope', 'Name')].astype(str).str.len()
result = pd.crosstab(
    index=df['name_len'],
    columns=df[('Assay', 'Response measured')]
)
result.columns = [f"{assay}" for assay in result.columns]
result.to_csv("dist_len_assay.csv")
#####################
df = df_clean[df_clean[('Epitope', 'Name')].str.len() == 15]
df['assay'] = df[('Assay', 'Method')].str.split('/').str[-1]
result_2 = pd.crosstab(
    index=df['assay'],
    columns=df[('Assay', 'Response measured')]
)
result_2.to_csv("dist_method_assay_dist_15mer.csv")

In [ ]:
import pandas as pd
import numpy as np

def clean_data_ic50(df):
    df = df[(df[('Assay', 'Response measured')].str.contains("IC50", regex=False, na=False, case=False))]
    df = df[~df[('Assay', 'Quantitative measurement')].isna()]
    df_clean = pd.DataFrame()
    df_clean['Epi_Seq'] = df[('Epitope', 'Name')]
    df_clean['HLA_Name_full'] = df[('MHC Restriction', 'Name')]
    df_clean['IC50'] = pd.to_numeric(df[('Assay', 'Quantitative measurement')], errors='coerce')
    df_clean['log_IC50'] = np.log10(df_clean['IC50'])
    df_clean = df_clean.groupby(['Epi_Seq', 'HLA_Name_full'])['log_IC50'].min().reset_index()
    df_clean['Target'] = (df_clean['log_IC50'] < np.log10(500)).astype(int)
    return df_clean

target_dict = {
    'Negative': 0,
    'Positive': 0.8,
    'Positive-Low': 0.3,
    'Positive-Intermediate': 0.6,
    'Positive-High': 1,
}

def target_filter(x):
    if str(x) in target_dict:
        return target_dict[x]
    else:
        return np.nan

def clean_data_qual(df):
    df_clean = pd.DataFrame()
    df_clean['Epi_Seq'] = df[('Epitope', 'Name')]
    df_clean['HLA_Name_full'] = df[('MHC Restriction', 'Name')]
    df_clean['Smooth_Target'] = df[('Assay', 'Qualitative Measurement')].apply(lambda x: target_filter(x)).values
    df_clean = df_clean.groupby(['Epi_Seq', 'HLA_Name_full'])['Smooth_Target'].max().reset_index()
    df_clean['Target'] = df_clean['Smooth_Target'].apply(lambda x: 1 if x >= 0.6 else 0)
    return df_clean

def clean_data_ms(df):
    df = df[(df[('Assay', 'Method')].str.contains('mass spectrometry'))]
    df_clean = pd.DataFrame()
    df_clean['Epi_Seq'] = df[('Epitope', 'Name')]
    df_clean['HLA_Name_full'] = df[('MHC Restriction', 'Name')]
    df_clean['Smooth_Target'] = df[('Assay', 'Qualitative Measurement')].apply(lambda x: target_filter(x)).values
    df_clean = df_clean.groupby(['Epi_Seq', 'HLA_Name_full'])['Smooth_Target'].max().reset_index()
    df_clean['Target'] = df_clean['Smooth_Target'].apply(lambda x: 1 if x >= 0.6 else 0)
    return df_clean

df_full = clean_data_ms(df_clean.copy())
df_neg_ic = clean_data_ic50(df_clean.copy())
df_neg_ic = df_neg_ic[df_neg_ic['Target'] == 0]
df_neg_ql = clean_data_qual(df_clean.copy())
df_neg_ql = df_neg_ql[df_neg_ql['Target'] == 0]
# df_full = clean_data_ic50(df_clean.copy())
df_full

In [ ]:
# df_full_conc = pd.concat([df_full, df_neg_ql], ignore_index=True)
# df_full_conc = df_full_conc.groupby(['Epi_Seq', 'HLA_Name_full'])['Target'].max().reset_index()
# df_full = df_full_conc

# 2. Dataset Filtering

### Human

In [ ]:
df_hum_15 = df_full[(df_full['HLA_Name_full'].str.contains("HLA")) & (df_full['Epi_Seq'].str.len() == 15)]
df_hum_15

In [ ]:
df_hum_15_pair = df_hum_15[(df_hum_15['HLA_Name_full'].str.contains("/"))].copy()
df_hum_15_pair_split = df_hum_15_pair['HLA_Name_full'].str.split('/', expand=True)
df_hum_15_pair['HLA_Name_A'] = df_hum_15_pair_split[0]
df_hum_15_pair['HLA_Name_B'] = 'HLA-' + df_hum_15_pair_split[1]

df_hum_15_tmp = df_hum_15[~(df_hum_15['HLA_Name_full'].str.contains("/"))]
df_hum_15_pair_dr = df_hum_15_tmp[df_hum_15_tmp['HLA_Name_full'].str.contains("DR")].copy()
df_hum_15_pair_dr['HLA_Name_B'] = df_hum_15_pair_dr['HLA_Name_full']
df_hum_15_pair_dr['HLA_Name_A'] = 'HLA-DRA*01:01'

df_hum_15_pair_merged = pd.concat([df_hum_15_pair, df_hum_15_pair_dr], ignore_index=True)
df_hum_15_pair_merged['HLA_Name'] = df_hum_15_pair_merged['HLA_Name_B'] + '_' + df_hum_15_pair_merged['HLA_Name_A']
idx = df_hum_15_pair_merged.groupby(['Epi_Seq', 'HLA_Name'])['Target'].idxmax()
df_hum_15_pair_merged = df_hum_15_pair_merged.loc[idx].reset_index(drop=True)
df_hum_15_pair_merged.to_csv("hum_pair_full.csv", index=False)
df_hum_15_pair_merged

In [ ]:
# df_hum_15_pair_merged['allele'] = df_hum_15_pair_merged['HLA_Name_B'].str.split('*', expand=True)[0].copy()
# df_hum_15_pair_merged_dist = df_hum_15_pair_merged.groupby('allele').size().reset_index(name='count')
# df_hum_15_pair_merged = df_hum_15_pair_merged.drop(columns=['allele'])
# df_hum_15_pair_merged_dist.to_csv("hla_dist_level0.csv", index=False)
# df_hum_15_pair_merged_dist

In [ ]:
# df_hum_15_pair_merged_dist_2 = df_hum_15_pair_merged.groupby('HLA_Name_B').size().reset_index(name='count')
# df_hum_15_pair_merged_dist_2 = df_hum_15_pair_merged_dist_2.sort_values(by='count', ascending=False)
# df_hum_15_pair_merged_dist_2.to_csv("hla_dist_level2.csv", index=False)
# df_hum_15_pair_merged_dist_2

In [ ]:
df_hum_15_sing = df_hum_15_tmp[~df_hum_15_tmp['HLA_Name_full'].str.contains("DR")].copy()
df_hum_15_sing['HLA_Name'] = df_hum_15_sing['HLA_Name_full']

df_hum_15_sing['HLA_Epi'] = df_hum_15_sing['HLA_Name'] + "_" + df_hum_15_sing['Epi_Seq']
df_hum_15_pair_merged['HLA_Epi'] = df_hum_15_pair_merged['HLA_Name_B'] + "_" + df_hum_15_pair_merged['Epi_Seq']
df_hum_15_sing_exclude = df_hum_15_sing[~df_hum_15_sing['HLA_Epi'].isin(df_hum_15_pair_merged['HLA_Epi'])]
df_hum_15_sing_exclude = df_hum_15_sing_exclude.drop(columns=['HLA_Epi'])
df_hum_15_pair_merged = df_hum_15_pair_merged.drop(columns=['HLA_Epi'])

df_hum_15_sing_exclude['HLA_Name_A'] = None
df_hum_15_sing_exclude['HLA_Name_B'] = None
df_hum_15_sing_exclude.loc[
    df_hum_15_sing_exclude['HLA_Name'].str.startswith('HLA-DQA1'),
    'HLA_Name_A'
] = df_hum_15_sing_exclude['HLA_Name']
df_hum_15_sing_exclude.loc[
    df_hum_15_sing_exclude['HLA_Name'].str.startswith(('HLA-DPB1', 'HLA-DQB1')),
    'HLA_Name_B'
] = df_hum_15_sing_exclude['HLA_Name']
df_hum_15_sing_exclude.to_csv("hum_sing_full.csv", index=False)
df_hum_15_sing_exclude

In [ ]:
df_hum_15_sing_exclude['allele'] = df_hum_15_sing_exclude['HLA_Name_full'].str.split('*', expand=True)[0].copy()
df_hum_15_sing_exclude_dist = df_hum_15_sing_exclude.groupby('allele').size().reset_index(name='count')
df_hum_15_sing_exclude = df_hum_15_sing_exclude.drop(columns=['allele'])
df_hum_15_sing_exclude_dist

### Animal

In [ ]:
df_ani_15 = df_full[~(df_full['HLA_Name_full'].str.contains("HLA")) & (df_full['Epi_Seq'].str.len() == 15)]
df_ani_15

In [ ]:
def get_hla_name_a(name):
    if name.startswith('BoLA'):
        return 'BoLA-DRA'
    elif name.startswith('Mamu'):
        return 'Mamu-DRA*01:01'
    elif name.startswith('SLA'):
        return 'SLA-DRA*01:01'
    elif name.startswith('H2'):
        return name + 'A'
    else:
        return None  # a default could go here
def get_hla_name_b(name):
    if name.startswith('H2'):
        return name + 'B'
    else:
        return None  # a default could go here
df_ani_15 = df_ani_15.dropna()
df_ani_15['HLA_Name_A'] = df_ani_15['HLA_Name_full'].apply(get_hla_name_a)
df_ani_15['HLA_Name_B'] = df_ani_15['HLA_Name_full'].apply(get_hla_name_b)

df_ani_15['HLA_Name'] = df_ani_15['HLA_Name_B'] + '_' + df_ani_15['HLA_Name_A']

df_ani_15.to_csv("ani_full.csv", index=False)
df_ani_15

## Concat

In [ ]:
df_hum_ani = pd.concat([df_hum_15_pair_merged, df_ani_15], ignore_index=True)
df_hum_ani_ic = pd.read_csv(P.QUALITATIVE_FULL)
df_hum_ani_ic = df_hum_ani_ic[df_hum_ani_ic['Target'] == 0]
df_hum_conc = pd.concat([df_hum_ani, df_hum_ani_ic], ignore_index=True)
idx = df_hum_conc.groupby(['Epi_Seq', 'HLA_Name'])['Target'].idxmax()
df_hum_conc = df_hum_conc.loc[idx].reset_index(drop=True)
df_hum_ani = df_hum_conc
df_hum_ani.to_csv("hum_ani_full.csv", index=False)
df_hum_ani

In [ ]:
epi_count = df_hum_ani['Epi_Seq'].unique()
epi_count = pd.DataFrame(epi_count, columns=['Epi_Seq'])
epi_count.to_csv("epi_count.csv", index=False)
len(epi_count)

In [ ]:
# df_hum_15_pair_merged['allele'] = df_hum_15_pair_merged['HLA_Name_full'].str.slice(0, 6)
# df_ani_15['allele'] = df_ani_15['HLA_Name_full'].str.slice(0, 5)
# df_hum_ani_tmp = pd.concat([df_hum_15_pair_merged, df_ani_15], ignore_index=True)
# df_hum_ani_tmp_dist = df_hum_ani_tmp.groupby('allele').size().reset_index(name='count')
# target_counts = df_hum_ani_tmp.groupby(['allele', 'Target']).size().unstack(fill_value=0)
# target_counts.columns = ['0_count', '1_count']
# target_counts['0_ratio'] = target_counts['0_count'] / (target_counts['0_count'] + target_counts['1_count'])
# target_counts['1_ratio'] = target_counts['1_count'] / (target_counts['0_count'] + target_counts['1_count'])
# df_hum_ani_tmp_dist = df_hum_ani_tmp_dist.merge(target_counts.reset_index(), on='allele')
# df_hum_15_pair_merged = df_hum_15_pair_merged.drop(columns=['allele'])
# df_ani_15 = df_ani_15.drop(columns=['allele'])
# df_hum_ani_tmp_dist.to_csv("mhc_dist_level0.csv", index=False)
# df_hum_ani_tmp_dist

In [ ]:
# df_hum_15_pair_merged['allele'] = df_hum_15_pair_merged['HLA_Name_B'].str.split('*', expand=True)[0].copy()
# df_ani_15['allele'] = df_ani_15['HLA_Name_full'].str.slice(0, 5)
# df_hum_ani_tmp = pd.concat([df_hum_15_pair_merged, df_ani_15], ignore_index=True)
# df_hum_ani_tmp_dist = df_hum_ani_tmp.groupby('allele').size().reset_index(name='count')
# target_counts = df_hum_ani_tmp.groupby(['allele', 'Target']).size().unstack(fill_value=0)
# target_counts.columns = ['0_count', '1_count']
# target_counts['0_ratio'] = target_counts['0_count'] / (target_counts['0_count'] + target_counts['1_count'])
# target_counts['1_ratio'] = target_counts['1_count'] / (target_counts['0_count'] + target_counts['1_count'])
# df_hum_ani_tmp_dist = df_hum_ani_tmp_dist.merge(target_counts.reset_index(), on='allele')
# df_hum_15_pair_merged = df_hum_15_pair_merged.drop(columns=['allele'])
# df_ani_15 = df_ani_15.drop(columns=['allele'])
# df_hum_ani_tmp_dist.to_csv("mhc_dist_level1.csv", index=False)
# df_hum_ani_tmp_dist

In [ ]:
df_hum_ani_dist_2 = df_hum_ani.groupby('HLA_Name_B').size().reset_index(name='count')
df_hum_ani_dist_2 = df_hum_ani_dist_2.sort_values(by='count', ascending=False)
target_counts = df_hum_ani.groupby(['HLA_Name_B', 'Target']).size().unstack(fill_value=0)
target_counts.columns = ['0_count', '1_count']
target_counts['0_ratio'] = target_counts['0_count'] / (target_counts['0_count'] + target_counts['1_count'])
target_counts['1_ratio'] = target_counts['1_count'] / (target_counts['0_count'] + target_counts['1_count'])
df_hum_ani_dist_2 = df_hum_ani_dist_2.merge(target_counts.reset_index(), on='HLA_Name_B')
df_hum_ani_dist_2.to_csv("mhc_dist_level2.csv", index=False)
df_hum_ani_dist_2

In [ ]:
# df_hum_15_pair_merged['allele'] = df_hum_15_pair_merged['HLA_Name_A'].str.split('*', expand=True)[0].copy()
# df_ani_15['allele'] = df_ani_15['HLA_Name_full'].str.slice(0, 5)
# df_hum_ani_tmp = pd.concat([df_hum_15_pair_merged, df_ani_15], ignore_index=True)
# df_hum_ani_tmp_dist = df_hum_ani_tmp.groupby('allele').size().reset_index(name='count')
# target_counts = df_hum_ani_tmp.groupby(['allele', 'Target']).size().unstack(fill_value=0)
# target_counts.columns = ['0_count', '1_count']
# target_counts['0_ratio'] = target_counts['0_count'] / (target_counts['0_count'] + target_counts['1_count'])
# target_counts['1_ratio'] = target_counts['1_count'] / (target_counts['0_count'] + target_counts['1_count'])
# df_hum_ani_tmp_dist = df_hum_ani_tmp_dist.merge(target_counts.reset_index(), on='allele')
# df_hum_15_pair_merged = df_hum_15_pair_merged.drop(columns=['allele'])
# df_ani_15 = df_ani_15.drop(columns=['allele'])
# df_hum_ani_tmp_dist.to_csv("mhc_dist_level1_alpha.csv", index=False)
# df_hum_ani_tmp_dist

In [ ]:
df_hum_ani_dist_2 = df_hum_ani.groupby('HLA_Name_A').size().reset_index(name='count')
df_hum_ani_dist_2 = df_hum_ani_dist_2.sort_values(by='count', ascending=False)
target_counts = df_hum_ani.groupby(['HLA_Name_A', 'Target']).size().unstack(fill_value=0)
target_counts.columns = ['0_count', '1_count']
target_counts['0_ratio'] = target_counts['0_count'] / (target_counts['0_count'] + target_counts['1_count'])
target_counts['1_ratio'] = target_counts['1_count'] / (target_counts['0_count'] + target_counts['1_count'])
df_hum_ani_dist_2 = df_hum_ani_dist_2.merge(target_counts.reset_index(), on='HLA_Name_A')
df_hum_ani_dist_2.to_csv("mhc_dist_level2_alpha.csv", index=False)
df_hum_ani_dist_2

In [ ]:
df_hum_ani_dist_2 = df_hum_ani.groupby('HLA_Name').size().reset_index(name='count')
df_hum_ani_dist_2 = df_hum_ani_dist_2.sort_values(by='count', ascending=False)
target_counts = df_hum_ani.groupby(['HLA_Name', 'Target']).size().unstack(fill_value=0)
target_counts.columns = ['0_count', '1_count']
target_counts['0_ratio'] = target_counts['0_count'] / (target_counts['0_count'] + target_counts['1_count'])
target_counts['1_ratio'] = target_counts['1_count'] / (target_counts['0_count'] + target_counts['1_count'])
df_hum_ani_dist_2 = df_hum_ani_dist_2.merge(target_counts.reset_index(), on='HLA_Name')
df_hum_ani_dist_2.to_csv("mhc_dist_level2_pair.csv", index=False)
df_hum_ani_dist_2

# Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
SEED = 42
np.random.seed(SEED)


df_original = df_hum_ani.copy()

print(df_original.head())

# --- 2. identify the groups and take a representative Target ---
group_cols = ['HLA_Name_B', 'Epi_Seq']
df_grouped = df_original.groupby(group_cols).agg(
    max_target=('Target', 'max'),
    group_size=('HLA_Name_A', 'size')
).reset_index()

print(df_grouped.head())

# --- 3. stratified split, handling single-member classes ---

target_counts = df_grouped['max_target'].value_counts()

single_targets = target_counts[target_counts == 1].index.tolist()

if single_targets:

    is_singleton = df_grouped['max_target'].isin(single_targets)
    df_grouped_singletons = df_grouped[is_singleton]
    df_grouped_main = df_grouped[~is_singleton]
else:
    df_grouped_main = df_grouped
    df_grouped_singletons = pd.DataFrame(columns=df_grouped.columns)

# --- 4. group-level stratified split, single members excluded ---
if not df_grouped_main.empty and len(df_grouped_main) >= 2 :
    n_samples_main = len(df_grouped_main)
    n_test_samples = int(np.ceil(n_samples_main * 0.3)) # test_size=0.3

    main_target_counts = df_grouped_main['max_target'].value_counts()
    if (main_target_counts < 2).any():
         # if excluding single members still leaves one, stratification is impossible
         # fall back to an unstratified split
         train_groups_df_split, test_groups_df_split = train_test_split(
            df_grouped_main,
            test_size=0.3,
            random_state=42
        )
    elif n_test_samples < len(main_target_counts):
         # fall back to an unstratified split
         train_groups_df_split, test_groups_df_split = train_test_split(
            df_grouped_main,
            test_size=0.3,
            random_state=42
        )
    else:
         # run the stratified split
         train_groups_df_split, test_groups_df_split = train_test_split(
             df_grouped_main,
             test_size=0.3,
             stratify=df_grouped_main['max_target'], # stratify on the target of the data being split
             random_state=42
         )

    # --- 5. single-member groups go to train ---
    train_groups_df = pd.concat([train_groups_df_split, df_grouped_singletons], ignore_index=True)
    test_groups_df = test_groups_df_split

elif not df_grouped_main.empty:
    train_groups_df = pd.concat([df_grouped_main, df_grouped_singletons], ignore_index=True)
    test_groups_df = pd.DataFrame(columns=df_grouped.columns)

else:
    train_groups_df = df_grouped_singletons
    test_groups_df = pd.DataFrame(columns=df_grouped.columns)


# --- 6. build the train/test group ids and assign the rows ---
# (earlier code omitted)
train_group_identifiers = set(zip(train_groups_df['HLA_Name_B'], train_groups_df['Epi_Seq']))
test_group_identifiers = set(zip(test_groups_df['HLA_Name_B'], test_groups_df['Epi_Seq']))


original_identifiers = list(zip(df_original['HLA_Name_B'], df_original['Epi_Seq']))
is_train = [identifier in train_group_identifiers for identifier in original_identifiers]

df_train_full = df_original[is_train].reset_index(drop=True)
if not all(is_train):
    df_test_full = df_original[~np.array(is_train)].reset_index(drop=True)
else:
    df_test_full = pd.DataFrame(columns=df_original.columns)

print(df_train_full.head())

if not df_test_full.empty:
    print(df_test_full.head())
else:
    pass

if test_group_identifiers:
    test_group_sample = list(test_group_identifiers)[0]

    is_in_train = any(row_tuple == test_group_sample
                      for row_tuple in zip(df_train_full['HLA_Name_B'], df_train_full['Epi_Seq']))

    is_in_test = any(row_tuple == test_group_sample
                     for row_tuple in zip(df_test_full['HLA_Name_B'], df_test_full['Epi_Seq']))


else:
    pass


# --- 7. dataset with HLA_Name_A dropped ---
df_train_no_A = df_train_full.drop(columns=['HLA_Name_A']).copy()
df_test_no_A = df_test_full.drop(columns=['HLA_Name_A']).copy()

# --- 8. de-duplicate after dropping HLA_Name_A ---
df_train_merged = df_train_no_A.sort_values('Target', ascending=False)\
                               .drop_duplicates(subset=['HLA_Name_B', 'Epi_Seq'], keep='first')\
                               .sort_index()

df_test_merged = df_test_no_A.sort_values('Target', ascending=False)\
                              .drop_duplicates(subset=['HLA_Name_B', 'Epi_Seq'], keep='first')\
                              .sort_index()

print(df_train_merged.head())

if not df_test_merged.empty:
    print(df_test_merged.head())
else:
    pass


# --- final dataset ---
# 1. df_train_full, df_test_full : using both A and B
# 2. df_train_merged, df_test_merged : using B only


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import os

# --- settings ---
N_FOLDS = 5
OUTPUT_DIR = 'train_folds'
GROUP_COLS = ['HLA_Name_B', 'Epi_Seq']

os.makedirs(OUTPUT_DIR, exist_ok=True)

unique_train_groups = df_train_full[GROUP_COLS].drop_duplicates()
unique_train_group_list = unique_train_groups.apply(tuple, axis=1).tolist()

if not unique_train_group_list:
    exit()

# --- 2. build the K-fold splitter over the group list ---
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# --- 3. split and write each fold ---
fold_counter = 0
for i, (_, val_indices) in enumerate(kfold.split(unique_train_group_list)):
    fold_counter += 1

    groups_in_fold = [unique_train_group_list[idx] for idx in val_indices]
    groups_in_fold_set = set(groups_in_fold)


    # --- 4. rows of df_train_full in this fold ---
    train_full_row_groups = list(zip(df_train_full[GROUP_COLS[0]], df_train_full[GROUP_COLS[1]]))
    is_in_fold_full = [group in groups_in_fold_set for group in train_full_row_groups]
    df_train_full_fold = df_train_full[is_in_fold_full].reset_index(drop=True)

    # --- 5. rows of df_train_merged in this fold ---
    train_merged_row_groups = list(zip(df_train_merged[GROUP_COLS[0]], df_train_merged[GROUP_COLS[1]]))
    is_in_fold_merged = [group in groups_in_fold_set for group in train_merged_row_groups]
    df_train_merged_fold = df_train_merged[is_in_fold_merged].reset_index(drop=True)


    # --- 6. write each fold to CSV ---
    full_filename = os.path.join(OUTPUT_DIR, f'train_pair_{i}_fold.csv')
    merged_filename = os.path.join(OUTPUT_DIR, f'train_beta_{i}_fold.csv')

    df_train_full_fold.to_csv(full_filename, index=False)
    df_train_merged_fold.to_csv(merged_filename, index=False)
    print(f"  - {full_filename}")
    print(f"  - {merged_filename}")

# --- done ---
if fold_counter == N_FOLDS:
    pass
else:
    pass

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

# --- assumes the variables from the previous step ---

unique_train_groups = df_train_full[GROUP_COLS].drop_duplicates()
unique_train_group_list = unique_train_groups.apply(tuple, axis=1).tolist()

if not unique_train_group_list:
    exit()

# --- 2. map each group to a fold number ---
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

group_to_fold_map = {}

for i, (_, val_indices) in enumerate(kfold.split(unique_train_group_list)):
    groups_in_fold = [unique_train_group_list[idx] for idx in val_indices]
    for group in groups_in_fold:
        group_to_fold_map[group] = i


# --- 3. add the Fold column to df_train_full ---
train_full_row_group_tuples = pd.Series(list(zip(df_train_full[GROUP_COLS[0]], df_train_full[GROUP_COLS[1]])))
df_train_full['Fold'] = train_full_row_group_tuples.map(group_to_fold_map).fillna(-1).astype(int)

# --- 4. add the Fold column to df_train_merged ---
train_merged_row_group_tuples = pd.Series(list(zip(df_train_merged[GROUP_COLS[0]], df_train_merged[GROUP_COLS[1]])))
df_train_merged['Fold'] = train_merged_row_group_tuples.map(group_to_fold_map).fillna(-1).astype(int)


# --- 5. check (optional) ---
print(df_train_full['Fold'].value_counts().sort_index())
if (df_train_full['Fold'] == -1).any():
    pass
else:
    pass

print(df_train_merged['Fold'].value_counts().sort_index())
if (df_train_merged['Fold'] == -1).any():
    pass
else:
    pass

print(df_train_full.head())
print(df_train_merged.head())

In [ ]:
df_train_merged = df_train_merged.groupby(['HLA_Name_B', 'Epi_Seq', 'Fold'])['Target'].max().reset_index()
df_train_merged['Target'] = df_train_merged['Target'].astype(int)
df_train_merged['HLA_Name'] = df_train_merged['HLA_Name_B']
df_train_merged.to_csv("train_beta.csv", index=False)

df_test_merged = df_test_merged.groupby(['HLA_Name_B', 'Epi_Seq', ])['Target'].max().reset_index()
df_test_merged['Target'] = df_test_merged['Target'].astype(int)
df_test_merged['HLA_Name'] = df_test_merged['HLA_Name_B']
df_test_merged.to_csv("test_beta.csv", index=False)

df_train_full.to_csv("train.csv", index=False)
df_test_full.to_csv("test.csv", index=False)